In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/all_results.pkl
/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/solo_bert_artifacts.pkl
/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/weights/bertimbau_large__cb_focal/fold_2.pt
/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/weights/bertimbau_large__cb_focal/fold_0.pt
/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/weights/bertimbau_large__cb_focal/fold_3.pt
/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/weights/bertimbau_large__cb_focal/fold_4.pt
/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/weights/bertimbau_large__cb_focal/fold_1.pt
/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/weights/bertimbau_large__cb_focal/tokenizer/tokenizer.json
/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2/weights/bertimbau_large__cb_focal/tokenize

In [2]:
# # %% [markdown]
# # # SPR 2026 – Mammography BI-RADS Classification – Inference
# # Notebook de inferência para submissão no Kaggle.
# # Carrega pesos treinados (5-fold ensemble do BERTimbau Large) e aplica calibração Otimizada.

# # %%
# import os
# import re
# import pickle
# import numpy as np
# import pandas as pd
# from pathlib import Path

# import torch
# import torch.nn.functional as F
# from torch.utils.data import Dataset, DataLoader
# from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# # ═══════════════════════════════════════════════════════════════════════════════
# # CONFIG
# # ═══════════════════════════════════════════════════════════════════════════════
# MAX_LEN     = 512
# NUM_CLASSES = 7
# N_FOLDS     = 5
# BATCH_SIZE  = 16
# DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # Caminho para os arquivos exportados no treino (Ajustar se necessário)
# WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-v2/advanced_outputs_kaggle_2')
# if not WEIGHTS_BASE.exists():
#     WEIGHTS_BASE = Path('advanced_outputs')

# CONFIG_KEY = 'bertimbau_large__cb_focal'
# weights_dir = WEIGHTS_BASE / 'weights' / CONFIG_KEY

# print(f'Device: {DEVICE}')
# print(f'Weights base: {WEIGHTS_BASE}')
# print(f'Weights dir: {weights_dir}')

# # %%
# # ═══════════════════════════════════════════════════════════════════════════════
# # DATASET
# # ═══════════════════════════════════════════════════════════════════════════════
# class MammographyDataset(Dataset):
#     def __init__(self, texts, tokenizer, max_len=192):
#         self.texts     = texts
#         self.tokenizer = tokenizer
#         self.max_len   = max_len

#     def __len__(self):
#         return len(self.texts)

#     def __getitem__(self, idx):
#         encoding = self.tokenizer(
#             self.texts[idx],
#             max_length=self.max_len,
#             padding='max_length',
#             truncation=True,
#             return_tensors='pt',
#         )
#         item = {
#             'input_ids':      encoding['input_ids'].squeeze(0),
#             'attention_mask': encoding['attention_mask'].squeeze(0),
#         }
#         if 'token_type_ids' in encoding:
#             item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
#         return item


# # %%
# # ═══════════════════════════════════════════════════════════════════════════════
# # CALIBRATION FUNCTIONS
# # ═══════════════════════════════════════════════════════════════════════════════
# def temperature_scale(probs, temperature):
#     logits = np.log(probs + 1e-10)
#     scaled = logits / temperature
#     exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
#     return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

# def apply_thresholds(probs, thresholds):
#     adjusted = probs * np.array(thresholds)
#     return adjusted.argmax(axis=1)


# # %%
# # ═══════════════════════════════════════════════════════════════════════════════
# # INFERENCE FUNCTION
# # ═══════════════════════════════════════════════════════════════════════════════
# @torch.no_grad()
# def predict(model, loader):
#     model.eval()
#     all_probs = []
#     for batch in loader:
#         input_ids      = batch['input_ids'].to(DEVICE)
#         attention_mask = batch['attention_mask'].to(DEVICE)
#         kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
#         if 'token_type_ids' in batch:
#             kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
#         outputs = model(**kwargs)
#         probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
#         all_probs.extend(probs)
#     return np.array(all_probs)


# # %%
# # ═══════════════════════════════════════════════════════════════════════════════
# # LOAD TEST DATA
# # ═══════════════════════════════════════════════════════════════════════════════
# test_path = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
# if not test_path.exists():
#     test_path = Path('data/test.csv')

# test_df = pd.read_csv(test_path)
# test_texts = test_df['report'].tolist()

# INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
# ACHADOS_MARKERS     = ['achados:\n', 'achados:']
# COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

# def extract_section(text, start_markers, end_markers=None):
#     txt_lower = text.lower()
#     start_idx = -1
#     for m in start_markers:
#         idx = txt_lower.find(m)
#         if idx >= 0:
#             start_idx = idx + len(m)
#             break
#     if start_idx < 0:
#         return ''
#     if end_markers is None:
#         return text[start_idx:].strip()
#     end_idx = len(text)
#     for m in end_markers:
#         idx = txt_lower.find(m, start_idx)
#         if 0 < idx < end_idx:
#             end_idx = idx
#     return text[start_idx:end_idx].strip()

# def clean_text(text):
#     text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
#     text = re.sub(r'[\n\r\t]+', ' ', text)
#     text = re.sub(r' {2,}', ' ', text)
#     return text.strip()

# def build_input_text(report_text):
#     report = clean_text(report_text)
#     indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
#     achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
#     comparativa = extract_section(report, COMPARATIVA_MARKERS)
#     if len(achados) < 10:
#         return report
#     parts = []
#     if indicacao:   parts.append(f'Indicação: {indicacao}')
#     if achados:     parts.append(f'Achados: {achados}')
#     if comparativa: parts.append(f'Comparativa: {comparativa}')
#     return ' '.join(parts)

# test_texts = [build_input_text(t) for t in test_texts]
# print(f'Test samples loaded: {len(test_df)}')

# # %%
# # ═══════════════════════════════════════════════════════════════════════════════
# # RUN FOLD ENSEMBLE INFERENCE
# # ═══════════════════════════════════════════════════════════════════════════════
# print(f'Loading tokenizer from: {weights_dir / "tokenizer"}')
# tokenizer = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')

# dataset = MammographyDataset(test_texts, tokenizer, MAX_LEN)
# loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False,
#                      num_workers=0, pin_memory=True)

# test_probs = np.zeros((len(test_df), NUM_CLASSES))
# folds_loaded = 0

# # Caminho do Config Base do Modelo
# #config_path = '/kaggle/input/datasets/gabrielsavio/model-config/model_config'
# # O config.json está dentro da subpasta model_config dentro de weights_dir
# config_path = weights_dir / 'model_config'

# for fold in range(N_FOLDS):
#     weight_path = weights_dir / f'fold_{fold}.pt'
#     if not weight_path.exists():
#         print(f'Fold {fold} not found, skipping...')
#         continue

#     print(f'Loading fold {fold}...', end=' ')

#     config = AutoConfig.from_pretrained(config_path, num_labels=NUM_CLASSES)
#     model = AutoModelForSequenceClassification.from_config(config).to(DEVICE)

#     state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
#     model.load_state_dict(state_dict)

#     fold_probs = predict(model, loader)
#     test_probs += fold_probs
#     folds_loaded += 1
#     print(f'done (shape: {fold_probs.shape})')

#     del model, state_dict
#     torch.cuda.empty_cache()

# # Probabilidade média dos folds
# test_probs /= folds_loaded
# print(f'\n{folds_loaded} folds loaded successfully.')

# # %%
# # ═══════════════════════════════════════════════════════════════════════════════
# # APPLY OPTIMIZATIONS & SUBMISSION
# # ═══════════════════════════════════════════════════════════════════════════════
# artifacts_path = WEIGHTS_BASE / 'solo_bert_artifacts.pkl'
# print(f'\nLoading calibration artifacts from: {artifacts_path}')

# with open(artifacts_path, 'rb') as f:
#     solo_artifacts = pickle.load(f)

# best_temp = solo_artifacts['temperature']
# best_thresholds = solo_artifacts['thresholds']

# print(f'Applied Temperature: {best_temp:.4f}')
# print(f'Applied Thresholds: {best_thresholds}')

# # Passo 1: Aplicar calibração de temperatura nas probabilidades brutas do teste
# calibrated_probs = temperature_scale(test_probs, best_temp)

# # Passo 2: Multiplicar pelos thresholds da classe para forçar a predição otimizada
# predictions = apply_thresholds(calibrated_probs, best_thresholds)

# # Passo 3: Criar o arquivo de submissão do Kaggle
# submission = pd.DataFrame({
#     'ID': test_df['ID'],
#     'target': predictions,
# })

# submission.to_csv('/kaggle/working/submission.csv', index=False)

# print('\n=== SUBMISSION ===')
# print(submission.to_string(index=False))
# print(f'\nsubmission.csv saved ({len(submission)} rows)')

In [3]:
# %% [markdown]
# # SPR 2026 – Mammography BI-RADS Classification – Inference (Treino Direto)
# Notebook de inferência para submissão no Kaggle.
# Carrega um único modelo treinado diretamente e aplica calibração de validação.

# %%
import os
import re
import pickle
import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO
# ═══════════════════════════════════════════════════════════════════════════════
MAX_LEN          = 512
NUM_CLASSES      = 7
BATCH_SIZE       = 16
DEVICE           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Configurações do modelo salvo no Treino Direto
CONFIG_KEY       = 'bertimbau_large__cb_focal'
DIRECT_SAVE_FOLD = 0  # Índice que você usou na hora de salvar o treino

WEIGHTS_BASE = Path('/kaggle/input/datasets/gabrielsavio/model-v3/advanced_outputs_kaggle_3')
if not WEIGHTS_BASE.exists():
    WEIGHTS_BASE = Path('advanced_outputs')

weights_dir = WEIGHTS_BASE / 'weights' / CONFIG_KEY

print(f'Device: {DEVICE}')
print(f'Weights base: {WEIGHTS_BASE}')
print(f'Weights dir: {weights_dir}')


# %%
# ═══════════════════════════════════════════════════════════════════════════════
# DATASET & FUNÇÕES DE PRÉ-PROCESSAMENTO
# ═══════════════════════════════════════════════════════════════════════════════
class MammographyDataset(Dataset):
    def __init__(self, texts, tokenizer, max_len=192):
        self.texts     = texts
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        item = {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
        }
        if 'token_type_ids' in encoding:
            item['token_type_ids'] = encoding['token_type_ids'].squeeze(0)
        return item

INDICACAO_MARKERS   = ['indicação clínica:\n', 'indicação clínica:', 'indicação:', 'indicacao:']
ACHADOS_MARKERS     = ['achados:\n', 'achados:']
COMPARATIVA_MARKERS = ['análise comparativa:', 'analise comparativa:']

def extract_section(text, start_markers, end_markers=None):
    txt_lower = text.lower()
    start_idx = -1
    for m in start_markers:
        idx = txt_lower.find(m)
        if idx >= 0:
            start_idx = idx + len(m)
            break
    if start_idx < 0:
        return ''
    if end_markers is None:
        return text[start_idx:].strip()
    end_idx = len(text)
    for m in end_markers:
        idx = txt_lower.find(m, start_idx)
        if 0 < idx < end_idx:
            end_idx = idx
    return text[start_idx:end_idx].strip()

def clean_text(text):
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\x81\x8d\x8f\x90\x9d\xad]', '', text)
    text = re.sub(r'[\n\r\t]+', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    return text.strip()

def build_input_text(report_text):
    report = clean_text(report_text)
    indicacao   = extract_section(report, INDICACAO_MARKERS, ACHADOS_MARKERS)
    achados     = extract_section(report, ACHADOS_MARKERS, COMPARATIVA_MARKERS)
    comparativa = extract_section(report, COMPARATIVA_MARKERS)
    if len(achados) < 10:
        return report
    parts = []
    if indicacao:   parts.append(f'Indicação: {indicacao}')
    if achados:     parts.append(f'Achados: {achados}')
    if comparativa: parts.append(f'Comparativa: {comparativa}')
    return ' '.join(parts)


# %%
# ═══════════════════════════════════════════════════════════════════════════════
# FUNÇÕES DE CALIBRAÇÃO & INFERÊNCIA
# ═══════════════════════════════════════════════════════════════════════════════
def temperature_scale(probs, temperature):
    logits = np.log(probs + 1e-10)
    scaled = logits / temperature
    exp_scaled = np.exp(scaled - scaled.max(axis=1, keepdims=True))
    return exp_scaled / exp_scaled.sum(axis=1, keepdims=True)

def apply_thresholds(probs, thresholds):
    adjusted = probs * np.array(thresholds)
    return adjusted.argmax(axis=1)

@torch.no_grad()
def predict(model, loader):
    model.eval()
    all_probs = []
    for batch in loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        kwargs = dict(input_ids=input_ids, attention_mask=attention_mask)
        if 'token_type_ids' in batch:
            kwargs['token_type_ids'] = batch['token_type_ids'].to(DEVICE)
        
        outputs = model(**kwargs)
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.extend(probs)
    return np.array(all_probs)


# %%
# ═══════════════════════════════════════════════════════════════════════════════
# PREPARAR DADOS DE TESTE
# ═══════════════════════════════════════════════════════════════════════════════
test_path = Path('/kaggle/input/competitions/spr-2026-mammography-report-classification/test.csv')
if not test_path.exists():
    test_path = Path('data/test.csv')

test_df = pd.read_csv(test_path)
test_texts = [build_input_text(t) for t in test_df['report'].tolist()]
print(f'Test samples loaded: {len(test_df)}')

print(f'Loading tokenizer from: {weights_dir / "tokenizer"}')
tokenizer = AutoTokenizer.from_pretrained(weights_dir / 'tokenizer')

dataset = MammographyDataset(test_texts, tokenizer, MAX_LEN)
loader  = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)


# %%
# ═══════════════════════════════════════════════════════════════════════════════
# EXECUTAR INFERÊNCIA (SINGLE MODEL)
# ═══════════════════════════════════════════════════════════════════════════════
weight_path = weights_dir / f'fold_{DIRECT_SAVE_FOLD}.pt'
config_path = weights_dir / 'model_config'

print(f'Carregando modelo: {weight_path}')
config = AutoConfig.from_pretrained(config_path, num_labels=NUM_CLASSES)
model = AutoModelForSequenceClassification.from_config(config).to(DEVICE)

state_dict = torch.load(weight_path, map_location=DEVICE, weights_only=True)
model.load_state_dict(state_dict)

print('Gerando predições brutas...')
test_probs = predict(model, loader)

del model, state_dict
torch.cuda.empty_cache()


# %%
# ═══════════════════════════════════════════════════════════════════════════════
# APLICAR CALIBRAÇÃO & GERAR SUBMISSÃO
# ═══════════════════════════════════════════════════════════════════════════════
# Atualizado para buscar o arquivo exato gerado pelo Treino Direto
artifacts_name = f'direct_artifacts_{CONFIG_KEY}_fold_{DIRECT_SAVE_FOLD}.pkl'
artifacts_path = WEIGHTS_BASE / artifacts_name

print(f'\nCarregando artefatos de calibração: {artifacts_path}')
with open(artifacts_path, 'rb') as f:
    direct_artifacts = pickle.load(f)

best_temp = direct_artifacts['temperature']
best_thresholds = direct_artifacts['thresholds']

print(f'Temperature Aplicada: {best_temp:.4f}')
print(f'Thresholds Aplicados: {[round(t, 3) for t in best_thresholds]}')

# 1. Aplicar calibração de temperatura nas probabilidades brutas do teste
calibrated_probs = temperature_scale(test_probs, best_temp)

# 2. Multiplicar pelos thresholds da classe para forçar a predição otimizada
predictions = apply_thresholds(calibrated_probs, best_thresholds)

# 3. Criar o arquivo de submissão do Kaggle
submission = pd.DataFrame({
    'ID': test_df['ID'],
    'target': predictions,
})

submission.to_csv('/kaggle/working/submission.csv', index=False)

print('\n=== SUBMISSION ===')
print(submission.head())
print(f'\nsubmission.csv saved ({len(submission)} rows)')

Device: cuda
Weights base: /kaggle/input/datasets/gabrielsavio/model-v3/advanced_outputs_kaggle_3
Weights dir: /kaggle/input/datasets/gabrielsavio/model-v3/advanced_outputs_kaggle_3/weights/bertimbau_large__cb_focal
Test samples loaded: 4
Loading tokenizer from: /kaggle/input/datasets/gabrielsavio/model-v3/advanced_outputs_kaggle_3/weights/bertimbau_large__cb_focal/tokenizer
Carregando modelo: /kaggle/input/datasets/gabrielsavio/model-v3/advanced_outputs_kaggle_3/weights/bertimbau_large__cb_focal/fold_0.pt
Gerando predições brutas...

Carregando artefatos de calibração: /kaggle/input/datasets/gabrielsavio/model-v3/advanced_outputs_kaggle_3/direct_artifacts_bertimbau_large__cb_focal_fold_0.pkl
Temperature Aplicada: 1.0371
Thresholds Aplicados: [np.float64(1.0), np.float64(0.5), np.float64(1.05), 1.0, np.float64(0.3), 1.0, np.float64(0.1)]

=== SUBMISSION ===
      ID  target
0   Acc0       6
1   Acc2       2
2   Acc4       1
3  Acc10       2

submission.csv saved (4 rows)
